In [1]:
import os
import pandas as pd
import json

In [2]:
# Define the expected columns
expected_columns = {
    "Scene", "Scenario", "Purpose", "EngineScript_sampleInterval",
    "TaskScript_start", "TaskScript_end", "TaskScript_pointsOfInterest",
    "TaskScript_numberOfAgents", "TaskScript_spawnInterval", "TaskScript_taskName",
    "TaskScript_agentType", "TaskScript_numberOfNeeds", "TaskScript_agentSpeed",
    "TaskScript_agentSize", "TaskScript_agentRadius", "TaskScript_privacyRadius",
    "TaskScript_poiTime", "TaskScript_revisit", "TaskScript_chooseNonDeterministically"
}

In [3]:
# Directory containing the CSV files
csv_dir = "Simulation setup 202503"
output_base_dir = "output_json_TEST"
os.makedirs(output_base_dir, exist_ok=True)

In [4]:

# Process each CSV file
for filename in os.listdir(csv_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(csv_dir, filename)
        df = pd.read_csv(filepath)
        
        # Check if all required columns are present
        if not expected_columns.issubset(df.columns):
            missing = expected_columns - set(df.columns)
            print(f"Skipping {filename}, missing columns: {missing}")
            continue
        
#         # Check if there are extra columns
#         if not set(df.columns).issubset(expected_columns):
#             extra_columns = set(df.columns) - expected_columns
#             print(f"# Skipping {filename}, extra columns: {extra_columns}")
#             continue 
        
        # Extract the prefix (e.g., PT1) from the filename
        prefix = filename.split("-")[0].strip()
        output_dir = os.path.join(output_base_dir, prefix)
        os.makedirs(output_dir, exist_ok=True)
        
        if 'Simulation run' not in df.columns:
            # Save each row as a JSON file
            for idx, row in df.iterrows():
                row_dict = row.to_dict()
                row_dict['idx']=idx

                json_filename = os.path.join(output_dir, f"{prefix}simId_{idx}_sampleNum_0.json")
                with open(json_filename, "w") as f:
                    json.dump(row_dict, f, indent=2)
        else:
            for idx, row in df.iterrows():
                num_sims=row['Simulation run']
                del row['Simulation run']
                for idy in range(num_sims):
                    row_dict = row.to_dict()
                    row_dict['idx']=idx
                    row_dict['idy']=idy
                    json_filename = os.path.join(output_dir, f"{prefix}_simId_{idx}_sampleNum_{idy}.json")
                    with open(json_filename, "w") as f:
                        json.dump(row_dict, f, indent=2)
                    
                    

print("Processing complete.")

Processing complete.
